# PDF Text Extraction with Bounding Boxes and JSON Output

## Objective
Extracts words, lines, field candidates, and fee-table entries from a loan worksheet while preserving bounding-box coordinates.

## Approach
- Read the PDF with PyMuPDF
- Group words into lines using block and line numbers
- Extract key fields and fee-table rows
- Export structured JSON with text and coordinates

## Expected Result
This notebook demonstrates structured document extraction, where raw PDF text is converted into JSON that can be used by downstream models or review tools.

## Running in Google Colab
These notebooks were developed in Google Colab. For reproducibility, place required PDFs in a `data/` folder when running locally, or upload them to the Colab working directory. The helper functions below try common Colab and GitHub-style paths.

## Security Note
API keys are not stored in the notebook. Use Colab Secrets with the name `GOOGLE_API_KEY` or set the environment variable manually.

## Project Context
This notebook is part of a curated document intelligence externship portfolio project completed through Outamation. The work focuses on OCR, document parsing, retrieval, LLM-based question answering, and prototype application development for mortgage-style document analysis.

## Data Note
The notebooks were originally developed in Google Colab. Any document files used for testing should be placed in the `data/` folder or uploaded directly into the Colab runtime. The sample documents used for this educational project do not contain sensitive personal information.


In [ ]:
# -------------------------
# PORTABLE COLAB/GITHUB HELPERS
# -------------------------
from pathlib import Path
import os

def resolve_path(filename_or_path):
    """Find a file in common Colab and GitHub project locations."""
    candidates = [
        Path(filename_or_path),
        Path("/content") / filename_or_path,
        Path("data") / Path(filename_or_path).name,
        Path("/content/data") / Path(filename_or_path).name,
    ]
    for path in candidates:
        if path.exists():
            return str(path)
    # Return GitHub-style path as the default so users know where to place data.
    return str(Path("data") / Path(filename_or_path).name)

def get_google_api_key():
    """Load GOOGLE_API_KEY from Colab Secrets or environment variables."""
    try:
        from google.colab import userdata
        key = userdata.get("GOOGLE_API_KEY")
        if key:
            os.environ["GOOGLE_API_KEY"] = key
            return key
    except Exception:
        pass
    return os.getenv("GOOGLE_API_KEY")

# Install required library
!pip install pymupdf -q

import fitz  # PyMuPDF
import json
import re
from collections import defaultdict

# -----------------------------
# 1. OPEN PDF
# -----------------------------
pdf_path = resolve_path("LenderFeesWorksheetNew (1).pdf")
doc = fitz.open(pdf_path)

# -----------------------------
# 2. EXTRACT WORDS + BOUNDING BOXES
# -----------------------------
all_words = []
all_lines = []

for page_num in range(len(doc)):
    page = doc[page_num]
    words = page.get_text("words")  # (x0, y0, x1, y1, text, block_no, line_no, word_no)

    line_groups = defaultdict(list)

    for w in words:
        x0, y0, x1, y1, text, block_no, line_no, word_no = w

        word_item = {
            "page": page_num + 1,
            "text": text,
            "bbox": [x0, y0, x1, y1],
            "block_no": block_no,
            "line_no": line_no,
            "word_no": word_no
        }
        all_words.append(word_item)

        line_groups[(block_no, line_no)].append((x0, y0, x1, y1, text, word_no))

    for (block_no, line_no), items in line_groups.items():
        items = sorted(items, key=lambda x: x[5])
        line_text = " ".join(item[4] for item in items)
        x0 = min(item[0] for item in items)
        y0 = min(item[1] for item in items)
        x1 = max(item[2] for item in items)
        y1 = max(item[3] for item in items)

        line_item = {
            "page": page_num + 1,
            "text": line_text,
            "bbox": [x0, y0, x1, y1],
            "block_no": block_no,
            "line_no": line_no
        }
        all_lines.append(line_item)

# Sort lines top-to-bottom, left-to-right
all_lines = sorted(all_lines, key=lambda x: (x["page"], x["bbox"][1], x["bbox"][0]))

# -----------------------------
# 3. HELPER FUNCTIONS
# -----------------------------
def find_line_containing(keyword):
    """Return first line containing keyword."""
    keyword_lower = keyword.lower()
    for line in all_lines:
        if keyword_lower in line["text"].lower():
            return line
    return None

def clean_money(text):
    match = re.search(r'[\$]?\s*([\d,]+\.\d+|[\d,]+)', text)
    return match.group(1) if match else None

def extract_field_from_line(line_text, field_name):
    """
    Extract value from patterns like:
    'Total Loan Amount: 380,000'
    'Interest Rate: 4.250 %'
    """
    pattern = re.escape(field_name) + r"\s*:?\s*(.+)"
    m = re.search(pattern, line_text, flags=re.IGNORECASE)
    return m.group(1).strip() if m else None

# -----------------------------
# 4. EXTRACT KEY FIELDS
# -----------------------------
key_fields = []

# Since this PDF is structured visually, some fields are easier to get
# by using known lines from the extracted text.

for line in all_lines:
    txt = line["text"]

    if "Applicants:" in txt:
        key_fields.append({
            "field": "Applicants",
            "text": txt,
            "bbox": line["bbox"]
        })

    elif "Date Prepared:" in txt:
        key_fields.append({
            "field": "Date Prepared",
            "text": txt,
            "bbox": line["bbox"]
        })

    elif "Loan Program:" in txt:
        key_fields.append({
            "field": "Loan Program",
            "text": txt,
            "bbox": line["bbox"]
        })

    elif "Prepared By:" in txt:
        key_fields.append({
            "field": "Prepared By",
            "text": txt,
            "bbox": line["bbox"]
        })

    elif "Total Loan Amount:" in txt and "Interest Rate:" in txt:
        key_fields.append({
            "field": "Loan Summary Line",
            "text": txt,
            "bbox": line["bbox"]
        })

    elif "TOTAL ESTIMATED FUNDS NEEDED TO CLOSE:" in txt:
        key_fields.append({
            "field": "Funds Needed To Close Section",
            "text": txt,
            "bbox": line["bbox"]
        })

    elif "TOTAL ESTIMATED MONTHLY PAYMENT:" in txt:
        key_fields.append({
            "field": "Monthly Payment Section",
            "text": txt,
            "bbox": line["bbox"]
        })

# Also capture important concrete values from lines that visually hold them
important_values = []
for line in all_lines:
    txt = line["text"]

    if "30 YEAR FIXED" in txt.upper():
        important_values.append({
            "field": "Loan Program Value",
            "text": txt,
            "bbox": line["bbox"]
        })

    elif "4.250" in txt and "360" in txt:
        important_values.append({
            "field": "Interest Rate / Term",
            "text": txt,
            "bbox": line["bbox"]
        })

    elif "John Q. Smith" in txt or "Mary A. Smith" in txt:
        important_values.append({
            "field": "Applicant Name Value",
            "text": txt,
            "bbox": line["bbox"]
        })

    elif "XYZ Lender" == txt.strip():
        important_values.append({
            "field": "Prepared By Value",
            "text": txt,
            "bbox": line["bbox"]
        })

# -----------------------------
# 5. EXTRACT TABLE DATA
# -----------------------------
# We will extract fee rows from ORIGINATION CHARGES and OTHER CHARGES.
# The logic here uses line text plus a regex that captures the amount at the end.

section = None
fee_rows = []

money_pattern = re.compile(r'\$\s*([\d,]+\.\d{2})')

for line in all_lines:
    txt = line["text"].strip()

    if txt.upper() == "ORIGINATION CHARGES":
        section = "Origination Charges"
        continue

    if txt.upper() == "OTHER CHARGES":
        section = "Other Charges"
        continue

    # stop once summary area begins
    if "TOTAL ESTIMATED FUNDS NEEDED TO CLOSE" in txt.upper():
        section = None

    if section:
        # Look for fee lines with a dollar amount
        if "$" in txt and not txt.upper().startswith("FEE PAID TO PAID BY"):
            matches = list(money_pattern.finditer(txt))
            if matches:
                last_match = matches[-1]
                amount = last_match.group(1)
                fee_text = txt[:last_match.start()].strip()

                fee_rows.append({
                    "section": section,
                    "text": txt,
                    "fee_description": fee_text,
                    "amount": amount,
                    "bbox": line["bbox"]
                })

# -----------------------------
# 6. OPTIONAL: PICK AT LEAST SIX KEY FIELDS CLEANLY
# -----------------------------
# These are the "important fields" you can mention in your submission.
selected_key_fields = [
    "Applicants",
    "Date Prepared",
    "Loan Program Value",
    "Prepared By Value",
    "Interest Rate / Term",
    "Funds Needed To Close Section",
    "Monthly Payment Section"
]

combined_key_output = []

for item in key_fields + important_values:
    if item["field"] in selected_key_fields:
        combined_key_output.append(item)

# Remove duplicates while preserving order
seen = set()
deduped_key_output = []
for item in combined_key_output:
    key = (item["field"], item["text"])
    if key not in seen:
        seen.add(key)
        deduped_key_output.append(item)

# -----------------------------
# 7. SAVE OUTPUTS
# -----------------------------
os.makedirs("outputs", exist_ok=True)
with open("outputs/all_words_with_bboxes.json", "w", encoding="utf-8") as f:
    json.dump(all_words, f, indent=2)

with open("outputs/all_lines_with_bboxes.json", "w", encoding="utf-8") as f:
    json.dump(all_lines, f, indent=2)

with open("outputs/key_fields_extracted.json", "w", encoding="utf-8") as f:
    json.dump(deduped_key_output, f, indent=2)

with open("outputs/fee_table_extracted.json", "w", encoding="utf-8") as f:
    json.dump(fee_rows, f, indent=2)

# -----------------------------
# 8. PRINT RESULTS
# -----------------------------
print("=" * 80)
print("KEY FIELDS EXTRACTED")
print("=" * 80)
for item in deduped_key_output:
    print(f"Field: {item['field']}")
    print(f"Text : {item['text']}")
    print(f"BBox : {item['bbox']}")
    print("-" * 80)

print("\n" + "=" * 80)
print("TABLE FIELDS EXTRACTED")
print("=" * 80)
for row in fee_rows:
    print(f"Section         : {row['section']}")
    print(f"Fee Description : {row['fee_description']}")
    print(f"Amount          : {row['amount']}")
    print(f"Full Text       : {row['text']}")
    print(f"BBox            : {row['bbox']}")
    print("-" * 80)

print("\n" + "=" * 80)
print("FILES SAVED")
print("=" * 80)
print("outputs/all_words_with_bboxes.json")
print("outputs/all_lines_with_bboxes.json")
print("outputs/key_fields_extracted.json")
print("outputs/fee_table_extracted.json")

# -----------------------------
# 9. SAMPLE STRUCTURED OUTPUT PREVIEW
# -----------------------------
sample_output = []

for item in deduped_key_output[:3]:
    sample_output.append({
        "text": item["text"],
        "bbox": item["bbox"]
    })

for row in fee_rows[:5]:
    sample_output.append({
        "text": f"{row['fee_description']} - ${row['amount']}",
        "bbox": row["bbox"]
    })

print("\n" + "=" * 80)
print("SAMPLE OUTPUT")
print("=" * 80)
print(json.dumps(sample_output, indent=2))
